(course-biophysics-conformational-engineering)=
# Module 27: Conformational Engineering

Channel gating motivates studying backbone changes. Use the original
[KcsA complex 1K4C](https://www.rcsb.org/structure/1K4C) to perturb the phi angle
at **chain C, group ID 90 (CYS)** by 10 degrees. Explicit chain labels avoid
selecting unrelated groups with the same ID in the complex. This illustrative
perturbation is not a validated gate-opening pathway or a demonstrated hinge.

> **🎯 Learning Outcomes**
> - Locate a backbone quartet across adjacent amino acids.
> - Choose a moving covalent block explicitly.
> - Preserve the input and verify a circular angular increment.

In [1]:
import numpy as np
import molsysmt as msm
from molsysmt import pyunitwizard as puw

molsys = msm.convert(msm.systems['Potassium channel KcsA']['1k4c.pdb'])
original_coordinates = puw.get_value(msm.get(molsys, coordinates=True), to_unit='nm').copy()
nitrogen = msm.select(molsys, selection='chain_id=="C" and group_id=="90" and atom_name=="N"')[0]
phi_quartets = np.asarray(msm.chemical_states.get_dihedral_quartets(molsys, phi=True))
quartet = phi_quartets[phi_quartets[:, 1] == nitrogen][0]

(course-biophysics-conformational-engineering-rotation)=
## 1. Perturbing the Backbone

Discover all phi quartets first: restricting discovery to one group would omit
the previous group's carbonyl C. The fourth atom determines the moving side
of the central N-CA bond. The explicit block also prevents moving other
components. Here `pbc=False` operates on the deposited complex coordinates;
periodic systems require their own reconstruction and image convention.

In [2]:
before = msm.structure.get_dihedral_angles(molsys, dihedral_quartets=quartet, pbc=False)
components = msm.chemical_states.get_covalent_blocks(
    molsys, remove_bonds=quartet[1:3], output_type='sets')
moving = sorted(next(block for block in components if int(quartet[3]) in block))
edited = msm.structure.shift_dihedral_angles(
    molsys, dihedral_quartets=quartet, shifts='10 degrees',
    blocks=[moving], pbc=False)
after = msm.structure.get_dihedral_angles(edited, dihedral_quartets=quartet, pbc=False)
print('Phi before/after (degrees):', puw.get_value(before, to_unit='degrees'),
      puw.get_value(after, to_unit='degrees'))

Phi before/after (degrees): [[-64.77748909]] [[-54.77748909]]


(course-biophysics-conformational-engineering-challenge)=
## 🏆 Challenge: Checking Motion Scope

Compare coordinates outside `moving` before and after the operation. Which
parts of chain C rotate? Which remain fixed? Verify the increment modulo
360 degrees; visual inspection alone does not establish it.

:::{key-takeaway}
:class: dropdown
Explicit quartets identify the measured torsion; explicit moving blocks identify
what an edit moves. A verified angle is a geometric result, not physical validation.
:::

(course-biophysics-conformational-engineering-see-also)=
:::{seealso}
:class: dropdown
{func}`molsysmt.structure.get_dihedral_angles`,
{func}`molsysmt.structure.shift_dihedral_angles`,
{func}`molsysmt.chemical_states.get_covalent_blocks`,
and [Module 16: Structural Operations](../Common_Core/16_Structural_Operations.ipynb).
:::